<a href="https://colab.research.google.com/github/DhanadurgaLakshmi/AI-Engineering/blob/main/Production_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
customers=[
    {
        "id":1,"name":"Alice","active":True
    },
     {
        "id":2,"name":123,"active":False
    },
      {
        "id":2,"name":"Akash","active":False
    },
     {
        "id":3,"active":True
    },
     {
        "id":4,"name":"Bob","active":"yes"
    },
]

In [ ]:
import logging
from typing import TypedDict

class Customer(TypedDict):
  id: int
  name: str
  active: bool

def get_active_customers(customers: list[Customer] | None) -> list[Customer]:
  result: list[Customer] = [] # Initialize as an empty list
  if customers is None:
    return [] # Return empty list immediately if no customers
  for customer in customers:
    try:
      # Ensure 'active' is a boolean and 'name' exists and starts with 'A'
      if customer.get("active", False) and customer.get("name", "").startswith('A'):
        result.append(customer)
    except KeyError as e:
      # Changed to .get() for safer access, but keeping logging for educational purposes
      # if a direct KeyError were to somehow still occur (e.g., if .get() wasn't used).
      logging.error(f"customer has missing field '{e.args[0]}' for customer id : {customer.get('id', 'N/A')}")
      continue
  return result # Moved outside the loop to process all customers

# Assuming 'customers' variable is defined in another cell in the notebook
# logging.basicConfig(level=logging.INFO) # Uncomment to see log messages
print(get_active_customers(customers))


[{'id': 1, 'name': 'Alice', 'active': True}]


In [ ]:


import logging
from typing import TypedDict
class Customer(TypedDict):
  id:int
  name:str
  active:bool
logger=logging.getLogger(__name__)
def get_active_customers(customers:list[Customer] | None) -> list[Customer]:
  if customers is None:
    return []
  result:list[Customer] =[]

  for customer in customers:
    if  "active" not in customer or  "name" not in customer:
      logger.warning("missing field in customer record for customer id : %s"+ customer.get("id","N/A"))
      continue
    if not isinstance(customer["active"],bool):
      logger.warning("invalid active value for customer id : %s "+ customer.get("id","N/A"))
      continue
    if not isinstance(customer["name"],str):
      logger.warning("invalid name value for customer id : %s"+ customer.get("id","N/A"))
      continue
    if customer["active"] and customer["name"].startswith("A"):
      result.append(customer)
  return result
print(get_active_customers(customers))


[{'id': 1, 'name': 'Alice', 'active': True}]


In [8]:
import requests
import logging
import time
logger=logging.getLogger(__name__)

def get_customer(customer_id):
  try:
    response=requests.get(f"https://api.example.com/customers/{customer_id}",timeout=5)
    response.raise_for_status()
    return response.json()
  except requests.Timeout:
    logger.error("Customer API request timed out for customer id : %s",customer_id)
    return None
  except requests.HTTPError as e:
    logger.error("Customer API returned HTTP Error for customer id : %s",customer_id,e.response.status_code)
    return None
  except requests.ConnectionError as e:
    logger.error("Failed to connect to customer API for customer id : %s",customer_id,e.response.status_code)
    return None

print(get_customer(1))


ERROR:__main__:Failed to connect to customer API for customer id : 1


None


In [9]:

import requests
import logging
import time
logger=logging.getLogger(__name__)


def get_customer(customer_id):
  max_attempts=3
  time_in_sec=1
  for attempt in range(1,max_attempts+1):
    try:
      response=requests.get(f"https://api/example.com/customer/{customer_id}",timeout=5)
      response.raise_for_status()
      return response.json()
    except requests.Timeout:
      if(attempt<max_attempts):
        delay=2**(attempt-1)
        logger.warning(f"Customer api request timed out for customer id : %s /n Retrying ... attempt : %s, after : %s second",customer_id,attempt,delay)
        time.sleep(delay)
        continue
      else:
        logger.error("Maximum number of retries reached for get customer api for customer id : %s",customer_id)
        return None
    except requests.ConnectionError:
      if(attempt <max_attempts):
        delay=2**(attempt-1)
        logger.warning(f"Failed to connect to customer api for customer id : %s /n Retrying ... attempt : %s, after : %s second",customer_id,attempt,delay)
        time.sleep(delay)
        continue
      else:
        logger.error("Maximum number of retries reached for get customer api for customer id : %s",customer_id)
        return None
    except requests.HTTPError as e:
      if(e.response.status_code==500 and attempt<max_attempts):
        delay=2**(attempt-1)
        logger.warning(f"Internal server error for get customer api for customer id : %s /n Retrying ... attempt : %s, after : %s second",customer_id,attempt,delay)
        time.sleep(delay)
        continue
      elif(e.response.status_code==429 attempt<max_attempts):
        delay=int(e.response.headers.get("Retry-After","1"))
        logger.warning(f"get customer request getting too many request and reached the max rate limit /n Retrying ... attempt : %s, after : %s second",attempt,delay)
        continue
      else:
        logger.error("Maximum number of retries reached for get customer api for customer id : %s",customer_id)
        return None

print(get_customer(101))





ERROR:__main__:Failed to connect to customer api for customer id : 101 /n Retrying ... attempt : 1, after : 1 second
ERROR:__main__:Failed to connect to customer api for customer id : 101 /n Retrying ... attempt : 2, after : 2 second
ERROR:__main__:Failed to connect to customer api for customer id : 101 /n Retrying ... attempt : 3, after : 4 second
ERROR:__main__:Failed to connect to customer api for customer id : 101 /n Retrying ... attempt : 4, after : 8 second
ERROR:__main__:Failed to connect to customer api for customer id : 101 /n Retrying ... attempt : 5, after : 16 second
ERROR:__main__:Maximum number of retries reached for get customer api for customer id : 101


None
